# Установка моделей

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Установка токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1225.48it/s]


In [3]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

TokenizerType = TokenizersBackend | SentencePieceBackend

class Generation:
    def __init__(
        self,
        llm_1: GenerationMixin,
        llm_2: GenerationMixin,
        tok_1: TokenizerType,
        tok_2: TokenizerType,
        top_k: int,
    ):
        self.llm_1 = llm_1
        self.llm_2 = llm_2

        self.tok_1 = tok_1
        self.tok_2 = tok_2

        self.top_k = top_k

        self.device_1 = next(llm_1.parameters()).device
        self.device_2 = next(llm_2.parameters()).device

        self.chat_prefix_1: str | None = None
        self.chat_prefix_2: str | None = None

    @staticmethod
    def _build_chat_prefix(
        tokenizer: TokenizerType,
        user_message: str,
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": user_message,
            }
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    @staticmethod
    def _tokenize_prompt(
        tokenizer: TokenizerType,
        prompt: str,
        device: torch.device,
    ):
        return tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

    @staticmethod
    def _generate_log_probs(
        model: GenerationMixin,
        inputs,
    ) -> torch.Tensor:
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]

        return torch.log_softmax(
            logits.float(),
            dim=-1,
        )

    def _get_top_distribution(
        self,
        log_probs: torch.Tensor,
        tokenizer: TokenizerType,
        model_number: int,
    ) -> dict[str, float]:
        k = min(self.top_k, log_probs.shape[-1])

        top_log_probs, top_token_ids = torch.topk(
            log_probs[0],
            k=k,
        )

        distribution: dict[str, float] = {}

        print(f"Модель {model_number}")

        for rank, (token_id, log_prob) in enumerate(
            zip(top_token_ids, top_log_probs, strict=True),
            start=1,
        ):
            token_id_int = token_id.item()

            token_text = tokenizer.decode(
                [token_id_int],
                skip_special_tokens=False,
            )

            probability = log_prob.exp().item()

            print(
                f"{rank}. "
                f"token={token_text!r}, "
                f"id={token_id_int}, "
                f"prob={probability:.6f}"
            )

            # Несколько ID теоретически могут декодироваться
            # в одну строку, поэтому вероятности складываем.
            distribution[token_text] = (
                distribution.get(token_text, 0.0)
                + probability
            )

        return distribution

    def initialize_chat(self, user_message: str) -> None:
        """
        Вызывается один раз перед генерационным циклом.
        """

        self.chat_prefix_1 = self._build_chat_prefix(
            tokenizer=self.tok_1,
            user_message=user_message,
        )

        self.chat_prefix_2 = self._build_chat_prefix(
            tokenizer=self.tok_2,
            user_message=user_message,
        )

        print("Конец chat-prefix модели 1:")
        print(repr(self.chat_prefix_1[-150:]))

        print("Конец chat-prefix модели 2:")
        print(repr(self.chat_prefix_2[-150:]))

    def generate_pipe(
        self,
        generated_text: str,
        model_to_freeze: str = "",
    ):
        """
        generated_text — только текст, сгенерированный ансамблем.
        Исходный пользовательский запрос сюда больше не передаётся.
        """

        if self.chat_prefix_1 is None or self.chat_prefix_2 is None:
            raise RuntimeError(
                "Сначала вызови initialize_chat(user_message)"
            )

        distributions: dict[str, dict[str, float]] = {}

        if model_to_freeze != "1":
            prompt_1 = self.chat_prefix_1 + generated_text

            inputs_1 = self._tokenize_prompt(
                tokenizer=self.tok_1,
                prompt=prompt_1,
                device=self.device_1,
            )

            log_probs_1 = self._generate_log_probs(
                model=self.llm_1,
                inputs=inputs_1,
            )

            distributions["1"] = self._get_top_distribution(
                log_probs=log_probs_1,
                tokenizer=self.tok_1,
                model_number=1,
            )

        if model_to_freeze != "2":
            prompt_2 = self.chat_prefix_2 + generated_text

            inputs_2 = self._tokenize_prompt(
                tokenizer=self.tok_2,
                prompt=prompt_2,
                device=self.device_2,
            )

            log_probs_2 = self._generate_log_probs(
                model=self.llm_2,
                inputs=inputs_2,
            )

            distributions["2"] = self._get_top_distribution(
                log_probs=log_probs_2,
                tokenizer=self.tok_2,
                model_number=2,
            )

        if "1" in distributions and "2" in distributions:
            return distributions["1"], distributions["2"]

        if "1" in distributions:
            return distributions["1"]

        if "2" in distributions:
            return distributions["2"]

        raise RuntimeError("Не была запущена ни одна модель")


In [4]:
import math

class Agreement:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.step_matrix = {}
        self.model_to_freeze = ''
        self.probs_generator = probs_generator
        self.input_str = input_str

    def runpipe(self):
        p = 0
        for p in range(10):
            self.step_matrix = {}
            self.prob_distribution_1, self.prob_distribution_2 = self.probs_generator.generate_pipe(input_str=self.input_str, 
                                                                                                    start_chat=True if p==0 else False,
                                                                                                    model_to_freeze=self.model_to_freeze)
            print(f'{self.prob_distribution_1=}')
            print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            self.match_tokens()
            max_key = max(self.step_matrix, key=self.step_matrix.get)
            self.input_str = self.input_str + max_key
            # freeze = self.model_to_freeze
            print(f'{self.step_matrix=}')
            print(f'очищаем step matrix')
            # print(f'{freeze=}')
        
    def sum_probs_common_token(self, token: str) -> float:
        return (math.log(self.prob_distribution_1[token]) + math.log(self.prob_distribution_2[token]))

    def exact_match_case(self, token) -> bool:
        if token in self.prob_distribution_2:
                self.step_matrix[token] = self.sum_probs_common_token(token=token)
                print(f'Нашли точное совпадение {token=} присутствует в обоих словарях')
                print(f'Матрица после суммирования вероятностей {self.step_matrix=}')
                return True

    def composite_match_case(self, token, opposite_distrib: dict[str, float], which_model_freeze: str) -> None:
        candidates = {}
        for elem in opposite_distrib:
            if elem.startswith(token) and token != elem:
                # self.model_to_freeze = which_model_freeze
                print(f'Элемент-кандидат: {token=}, во что он может выродится: {elem=}')
                if token not in candidates:                    
                    candidates[token] = [elem]
                else:
                    candidates[token].append(elem)

        print(f'Кандидатский словарь: {candidates=}')
        
        for key, value in candidates.items():
            print(f'Выполняем прогноз для {key=}')
            print(f'Строка-продолжение: {self.input_str+key}')
            tmp_distrib= self.probs_generator.generate_pipe(input_str=self.input_str+key, 
                                                                              start_chat=False,
                                                                              model_to_freeze=which_model_freeze)
            
            print(f'Генерация-наследник 1: {tmp_distrib=}')
            for key2 in tmp_distrib:
                # print(f'рассматриваем {key2=}')
                for val in value:
                    print(f'сравниваем {val=} с {key+key2=}')
                    comparison = key+key2
                    if comparison.startswith(val) and comparison != val:
                        print(f'Нашли префикс. {comparison=} начинается с {val=}')
                        self.step_matrix[val] += math.log(tmp_distrib[key2])
                        print(f'{self.step_matrix=}')
                    elif comparison == val:
                        print(f'Нашли точное совпадение в наследнике. Цикл поиска завершен. {comparison=} и {val=}')
                        if val not in self.step_matrix:
                            if key in self.prob_distribution_1:
                                self.step_matrix[val] = math.log(self.prob_distribution_1[key])+math.log(tmp_distrib[key2])
                            elif key in self.prob_distribution_2:
                                self.step_matrix[val] = math.log(self.prob_distribution_2[key])+math.log(tmp_distrib[key2])
                        print(f'{self.step_matrix}')
            print(f'################################')
                
    def match_tokens(self) -> None:
        for token_1 in self.prob_distribution_1:
            exact_match = self.exact_match_case(token_1)
            if not exact_match:
                self.composite_match_case(token_1, opposite_distrib=self.prob_distribution_2, which_model_freeze='1')
        for token_2 in self.prob_distribution_2:
            self.composite_match_case(token_2, opposite_distrib=self.prob_distribution_1, which_model_freeze='2')

In [13]:
from pprint import pprint

class Agreement2:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.step_matrix = {}
        self.model_to_freeze = ''
        self.max_steps: int = 10
        self.probs_generator = probs_generator
        self.user_prompt = input_str
        self.generated_text = ""
    
    def runpipe(self):
        self.probs_generator.initialize_chat(
            user_message=self.user_prompt,
        )

        p = 0
        print(f'{self.max_steps}')
        for step in range(self.max_steps):
            print("###############################################################")
            print(f"Шаг: {step}")
            print("Пользовательский запрос:")
            print(self.user_prompt)
            print("Сгенерированное продолжение:")
            print(repr(self.generated_text))
            print("###############################################################")
            result = self.probs_generator.generate_pipe(
                generated_text=self.generated_text,
                model_to_freeze=self.model_to_freeze,
            )
            self.prob_distribution_1, self.prob_distribution_2 = result
            print(f'{self.prob_distribution_1=}')
            print(f'{self.prob_distribution_2=}')
            ### Блок сопоставления токенов ###
            selected_text = self.match_chars()
            self.generated_text += selected_text
            print(f"Выбранный фрагмент: {selected_text!r}")
            print(f"Текущий ответ: {self.generated_text!r}")
        return self.generated_text

    def count_min_token_len_per_distrib(self, distribs: list):
        distrib_min_len = {}
        for idx, distrib in enumerate(distribs):
            distrib_min_len[idx] = min(set([len(key) for key in distrib]))
        return distrib_min_len

    def count_max_token_len_per_distrib(self, distribs: list):
        distrib_min_len = {}
        for idx, distrib in enumerate(distribs):
            distrib_min_len[idx] = max(set([len(key) for key in distrib]))
        return distrib_min_len

    def find_the_suitest_token(self, distrib: dict, char_num: int, ):
        char_prob = {}
        for token, probability in distrib.items():
            print(f'{char_num=}')
            print(f'{token=}')
            if token[char_num] not in char_prob:
                char_prob[token[char_num]] = probability
            else:
                char_prob[token[char_num]] += probability
        
        if len(char_prob) == len(distrib):
            return char_prob
        print(f'{char_prob=}')
        the_most_common_char = max(char_prob, key=char_prob.get)
        return the_most_common_char
    
    def ensemble(self, ensemble_distr: dict):
        token_prob = {}
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token

    def ensemble_str(self, ensemble_dict: dict, char_num: int, probs_list: list):
        ensembling_probs = {}
        print(f'здесь {ensemble_dict=}')
        for distr in ensemble_dict.values():
            print(f'{distr=}')
            for distribution in distr.values():
                if not isinstance(distribution, str):
                    for token, prob in distribution.items():
                        if token not in ensembling_probs:
                            ensembling_probs[token] = prob
                        else:
                            ensembling_probs[token] += prob
                else:
                    result = "".join(distr[key] for key in sorted(distr))
                    print(f'{result=}')
                    print(f'{probs_list=}')
                    for elem in probs_list:
                        for token, prob in elem.items():
                            if token.startswith(result):
                                print(f'Токен {token} начинается с {result}, добавляем его вероятность')
                                if result not in ensembling_probs:
                                    ensembling_probs[result] = prob
                                else:
                                    ensembling_probs[result] += prob
        
        pprint(f'{ensembling_probs=}')
        most_likely_token = max(ensembling_probs, key=ensembling_probs.get)
        return most_likely_token

    def match_chars(self):
        probs_list = [self.prob_distribution_1, self.prob_distribution_2]
        distrib_min_len = self.count_min_token_len_per_distrib(distribs=probs_list)
        distrib_max_len = self.count_max_token_len_per_distrib(distribs=probs_list)

        ensemble_dict = {}
        print(f'{probs_list=}')
        print(f'{distrib_min_len=}')
        print(f'{distrib_max_len=}')
        stop = False
        prefix = ''
        p = 0

        while p < 5:
            for char_num in distrib_max_len:
                for distrib_num, distrib in enumerate(probs_list):
                    the_most_common_char = self.find_the_suitest_token(distrib=distrib, char_num=char_num) 
                    print(f'{the_most_common_char=}')
                    print(f'Ансмабль дикт на старте {ensemble_dict=}')
                    if isinstance(the_most_common_char, dict): # Кейс, когда нет общих совпадений внутри модели
                        ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                    elif isinstance(the_most_common_char, str): # Кейс, когда есть общие совпадения внутри модели
                        if f'distrib_num_{distrib_num}' not in ensemble_dict:
                            ensemble_dict[f'distrib_num_{distrib_num}'] = {char_num: the_most_common_char}
                        else:
                            ensemble_dict[f'distrib_num_{distrib_num}'][char_num] = the_most_common_char

                print(f'Ансмабль дикт на финише {ensemble_dict=}')
                print(f'Финалисты {char_num}-того символа {ensemble_dict=}')
                most_likely_token = self.ensemble_str(ensemble_dict=ensemble_dict, char_num=char_num, probs_list=probs_list)
                prefix = prefix + most_likely_token
                print(f'{prefix=}')
                print(f'{most_likely_token=}')
                print(f'{probs_list=}')
                new_probs_list = []
                
                # Сортировка по релеватным для продолжения токенам
                print(f'Начинаем зачистку токенов, не начинающихся с {prefix}')
                pprint(f'Как было до: {probs_list=}')
                for elem in probs_list:
                    tmp_distrib = {}
                    for token, prob in elem.items():
                        if token.startswith(most_likely_token):
                            print(f'Токен {token} начинается с {most_likely_token}, оставляем его в пуле на следующую проверку')
                            tmp_distrib[token] = prob
                    new_probs_list.append(tmp_distrib)
                
                print(f'Как стало после: {new_probs_list=}')
                # print(f'{len(new_probs_list)=}')
                # print(f'{len(probs_list)=}')
                # for elem in new_probs_list:
                #     if elem.keys
                #         if 
                if all(current.keys() == new_probs_list[0].keys() for current in new_probs_list[1:]): # Проверка на идентичность ключей
                    for elem in new_probs_list:
                        for string in elem.keys():
                            return_string = string
                    print(f'Возвращаем {return_string}')
                    return return_string
                p += 1
                print(f'{new_probs_list=}')

            

                







        # for idx, distrib in enumerate(probs_list):
        #     for char_num in range(min(distrib_min_len.values())):
        #         the_most_common_char = self.find_the_suitest_token(distrib=distrib, char_num=char_num)
        #         print(f'Модель {idx}, символ под индексом {char_num}, наиболее популярный символ: {the_most_common_char}')
        #         if isinstance(the_most_common_char, dict):
        #             print('зашли сюда')
        #             distrib = the_most_common_char
        #             ensemble_dict[idx] = distrib
        #         elif isinstance(the_most_common_char, str):
        #             winning_char = the_most_common_char
        #             if idx not in ensemble_dict:
        #                 ensemble_dict[idx] = winning_char
        #             else:
        #                 ensemble_dict[idx] = ensemble_dict[idx]+winning_char
                    
        # print(f'{ensemble_dict=}')
        # if isinstanceensemble_dict.values()
        # for (model_num, token), (model_idx, prob) in  zip(ensemble_dict.items(), enumerate(probs_list)):
        #     print(f'{prob=}')
        #     if token in prob:
        #         if prob[token] < max(list(prob.values())):
        #             added_token = token
        #             new_distrib = self.probs_generator.generate_pipe(generated_text=self.generated_text+added_token,
        #                                                                       model_to_freeze=self.model_to_freeze,)
        #             print(f'{new_distrib=}')
        # the_most_popular_token = self.ensemble(ensemble_distr=ensemble_dict)
        # print(f'{the_most_popular_token=}')
        # return the_most_popular_token
            

In [14]:
agg = Generation(
    llm_1=model_1,
    llm_2=model_2,
    tok_1=tokenizer_1,
    tok_2=tokenizer_2,
    top_k=5,
)

agreement = Agreement2(
    probs_generator=agg,
    input_str=(
        "Natalia sold clips to 48 of her friends in April, "
        "and then she sold half as many clips in May. "
        "How many clips did Natalia sell altogether in April and May?"
    ),
)

answer = agreement.runpipe()

print("Итог:")
print(repr(answer))

Конец chat-prefix модели 1:
's in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n'
Конец chat-prefix модели 2:
's in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n'
10
###############################################################
Шаг: 0
Пользовательский запрос:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Сгенерированное продолжение:
''
###############################################################
Модель 1
1. token='N', id=555, prob=0.975372
2. token='In', id=1286, prob=0.020562
3. token='She', id=8052, prob=0.000862
4. token='Let', id=8232, prob=0.000523
5. token=' Natal', id=34750, prob=0.000348
Модель 2
1. token='To', id=1249, prob=0.844905
2. token='N', id=45, prob=0.0904

TypeError: can only concatenate str (not "NoneType") to str

In [ ]:
class Pipeline:
    def __init__(self, generator: Generation):
        self.generator = generator

    def runpipe(self, input_str: str):
        freeze = ''
        for p in range(1000):
            models_probs_list = self.generator.generate_pipe(input_str=input_str, start_chat=True if p==0 else False,
                                                             model_to_freeze=freeze)
            
            agreement = Agreement(prob_distribution_1=models_probs_list['1'],
                                  prob_distribution_2=models_probs_list['2'])

            agreement.match_tokens()
            max_key = max(agreement.step_matrix, key=agreement.step_matrix.get)
            input_str = input_str + max_key
            freeze = agreement.model_to_freeze
            print(f'{freeze=}')

In [ ]:
pipe = Pipeline(generator=agg)
pipe.runpipe(input_str="Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?")

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class GenerationLoop:
    def __init__(self, 
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizersBackend | SentencePieceBackend,
                 tokenizer_2: TokenizersBackend | SentencePieceBackend,
                 mapping_vocab: dict):

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class Benchmark:
    def __init__(self,
                 map_proj: dict, 
                 map_mlp: dict, 
                 llm_1: GenerationMixin, 
                 llm_2: GenerationMixin, 
                 tok_1: TokenizersBackend | SentencePieceBackend, 
                 tok_2: TokenizersBackend | SentencePieceBackend,):
        self.map_proj = map_proj
        self.map_mlp = map_mlp
        self.llm_1 = llm_1
        self.llm_2 = llm_2
        self.tok_1 = tok_1
        self.tok_2 = tok_2
    
    def projecting_loop(self, tokens):
        projected_idx_mlp = []
        projected_idx_proj = []
        for idx in tokens:
            token = self.tok_1.decode(idx)
            idx = int(idx)
            print(f'id "{idx}", строка "{token}"')
            if token == '\n': 
                token = 'Ċ'
            if token.startswith(' '):
                token = token.replace(' ',"Ġ")
            
            mlp_projection = self.map_mlp[(token, idx)]
            mtrx_projection = self.map_proj[(token, idx)]

            print(f'НС-отображение {mlp_projection}') 
            print(f'Проекционное отображение {mtrx_projection}') 
            
            projected_idx_mlp.append(mlp_projection[0][1])

            if len(mtrx_projection) > 1:
                for elem in mtrx_projection:
                    print(f'{elem=}')
                    projected_idx_proj.append(elem[1])
            else:
                projected_idx_proj.append(mtrx_projection[0][1])
            print('##########################################')
        

        # print(f'{projected_arr_mlp=}')
        # print(f'{projected_arr_proj=}')
        projected_only_idx_mlp = []
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_mlp)}')
        print(f'Декодированная строка {self.tok_2.decode(projected_idx_proj)}')


    def generation_pipe(self, msg: list):
        text = self.tok_1.apply_chat_template(msg,
                                              tokenize=True,
                                              return_tensors='pt',
                                              add_generation_prompt=False)
        
        tokens_1 = text['input_ids'][0]
        strings = self.tok_1.decode(tokens_1, skip_special_tokens=True)
        return tokens_1, strings

    def map_input(self, input: str):
        message = [{'role': 'user', 'content': input}]

        tokens, strings = self.generation_pipe(msg=message)

        print(f'Разбивка по токенам: {tokens=}')
        print(f'Декодировано в текст: {strings=}')

        self.projecting_loop(tokens)

In [ ]:
bench = Benchmark(map_proj=proj_mapping,
                  map_mlp=mlp_mapping,
                  llm_1=model_1,
                  llm_2=model_2,
                  tok_1=tokenizer_model_1,
                  tok_2=tokenizer_model_2)

bench.map_input(input="The file, 90 megabytes in size, downloads at the rate of 5 megabytes per second for its first 60 megabytes, and then 10 megabytes per second thereafter. How long, in seconds, does it take to download entirely?")